# Drive Mount & Environments

In [ ]:
# ✅ [환경 설정]
# Google Drive 마운트 + 필수 라이브러리 설치
# pretty_midi: MIDI 파싱 / tqdm: 진행률 / scikit-learn: train/val split

from google.colab import drive
drive.mount('/content/drive')

!pip install -q pretty_midi tqdm scikit-learn

# ✅ [임포트]
# 전처리 파이프라인에 필요한 모든 라이브러리 한 곳에 모아둠

import os, json, glob, bisect
from collections import defaultdict
from multiprocessing import Pool
import bisect  # as bs 없이 그냥 bisect으로 통일
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import pretty_midi

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 37.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.7 MB/s eta 0:00:00


In [ ]:
import shutil, tarfile, os, time, glob
from tqdm.auto import tqdm

DRIVE_TAR = '/content/drive/MyDrive/midi_gpt2_original/lmd_full.tar'
LOCAL_TAR = '/content/lmd_full.tar'
LOCAL_DIR = '/content/home/jhyim0823/GETMusic/lmd_full'  # ✅ 실제 해제 경로

# Step 1: tar 파일 Drive → 로컬 복사
if not os.path.exists(LOCAL_TAR):
    print(f"📦 tar 복사 중: {DRIVE_TAR} → {LOCAL_TAR}")
    start = time.time()
    shutil.copy2(DRIVE_TAR, LOCAL_TAR)
    elapsed = time.time() - start
    size_gb = os.path.getsize(LOCAL_TAR) / 1e9
    print(f"✅ 복사 완료: {size_gb:.1f}GB / {elapsed:.0f}초")
else:
    print(f"✅ 이미 존재: {LOCAL_TAR} (복사 스킵)")

# Step 2: tar 압축 해제
if not os.path.exists(LOCAL_DIR):
    print(f"📂 압축 해제 중: {LOCAL_TAR} → {LOCAL_DIR}")
    start = time.time()
    with tarfile.open(LOCAL_TAR, 'r') as tar:
        members = tar.getmembers()
        for member in tqdm(members, desc="압축 해제", unit="file"):
            tar.extract(member, path='/content', filter='data')  # ✅ DeprecationWarning 제거
    elapsed = time.time() - start
    print(f"✅ 압축 해제 완료: {elapsed:.0f}초")
else:
    print(f"✅ 이미 존재: {LOCAL_DIR} (해제 스킵)")

# Step 3: 확인
midi_count = len(glob.glob(f'{LOCAL_DIR}/**/*.mid', recursive=True))  # ✅ LOCAL_DIR 변수 재사용
print(f"\n✅ 로컬 MIDI 파일 수: {midi_count:,}개")
print(f"✅ 전처리 준비 완료 — Cell 10 실행하세요")

# Vocabulary Definition & MidiCaps Mapping

In [ ]:
# ✅ [Cell 4] Vocab 정의 — 변경 없음
def build_v5_vocab():
    vocab = {}
    def add(prefix, r):
        for i in r: vocab[f"{prefix}{i}"] = len(vocab)

    for t in ["PAD","BOS","EOS","SEP","PIECE_START","PIECE_END",
              "BAR_START","BAR_END","PHRASE_END",
              "<PRE>","<SUF>","<MID>"]:
        vocab[t] = len(vocab)

    for g in ["CLASSICAL","JAZZ","POP","ROCK","ELECTRONIC","FOLK","UNKNOWN"]:
        vocab[f"GENRE_{g}"] = len(vocab)

    roots = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
    for r in roots:
        for m in [":maj",":min"]: vocab[f"KEY_{r}{m}"] = len(vocab)
    vocab["KEY_NONE"] = len(vocab)

    for p in [40, 68, 73]: vocab[f"TARGET_{p}"] = len(vocab)

    for m in ["4:4","3:4","2:4","6:8","12:8","OTHER"]:
        vocab[f"METER_{m}"] = len(vocab)

    add("DENSITY_", range(1, 6))
    add("INST=",    range(129))
    for a in ["ART_NORMAL","ART_LEGATO","ART_VIBRATO","ART_STACCATO"]:
        vocab[a] = len(vocab)
    add("EXPR_",  range(32))
    add("TIME=",  range(96))
    add("PITCH=", range(128))
    add("DUR=",   range(1, 193))
    add("VEL=",   range(32))

    for w in ["melodic","epic","calm","fast","slow","sad","happy",
              "piano","strings","orchestra","cinematic"]:
        vocab[f"TEXT_{w}"] = len(vocab)

    return vocab

vocab = build_v5_vocab()
print(f"✅ Vocab 크기: {len(vocab):,}")

✅ Vocab 크기: 682


In [ ]:
# ✅ [MidiCaps 메타데이터 로더]
# MidiCaps JSONL에서 MD5 해시 기반으로 캡션 매핑
# 장르 추출 함수도 여기서 정의
# 학습만 할 때는 하지 말기 !

def load_midicaps_inventory(json_path):
    cap_map = {}
    with open(json_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            entry = json.loads(line)
            if 'location' not in entry: continue  # ✅ 추가
            file_key = os.path.splitext(
                os.path.basename(entry['location']))[0].lower().strip()
            cap_map[file_key] = entry
    print(f"✅ MidiCaps 로드 완료: {len(cap_map):,}개")
    return cap_map

def extract_genre_token(caption, vocab_dict):
    if not caption: return vocab_dict["GENRE_UNKNOWN"]
    cap = caption.lower()
    keywords = {
        "CLASSICAL": ['classical','orchestra'],
        "JAZZ":      ['jazz','blues'],
        "POP":       ['pop','ballad'],
        "ROCK":      ['rock','guitar'],
        "ELECTRONIC":['electronic','synth'],
        "FOLK":      ['folk','acoustic']
    }
    for g, keys in keywords.items():
        if any(k in cap for k in keys):
            return vocab_dict[f"GENRE_{g}"]
    return vocab_dict["GENRE_UNKNOWN"]

# Tokenizer

In [ ]:
# ✅ [Cell 7] V5 토크나이저 — _23_ 버그 수정 완료본
FLAT_TO_SHARP = {
    "Db": "C#", "Eb": "D#", "Fb": "E",
    "Gb": "F#", "Ab": "G#", "Bb": "A#", "Cb": "B"
}

CHUNK_SIZE   = 2048
CHUNK_STEP   = 1024
MIN_CHUNK    = 100
MIN_BODY_LEN = 50

def midi_to_v5_tokens(midi_path, vocab_dict, metadata):
    try:
        pm          = pretty_midi.PrettyMIDI(midi_path)
        res         = pm.resolution
        timeline    = defaultdict(lambda: defaultdict(list))

        key_changes = sorted(pm.key_signature_changes, key=lambda x: x.time)
        key_times   = [k.time for k in key_changes]
        ts_changes  = sorted(pm.time_signature_changes, key=lambda x: x.time)
        ts_times    = [t.time for t in ts_changes]

        tempo_change_times, tempos = pm.get_tempo_changes()

        caption     = metadata.get('caption', '').lower()
        genre_token = extract_genre_token(caption, vocab_dict)
        text_tokens = [vocab_dict[f"TEXT_{w}"]
                       for w in caption.replace(",", " ").split()
                       if f"TEXT_{w}" in vocab_dict][:10]

        found = False

        for inst in pm.instruments:
            p = 128 if inst.is_drum else inst.program

            e_ch = sorted([cc for cc in inst.control_changes if cc.number == 11],
                          key=lambda x: x.time)
            m_ch = sorted([cc for cc in inst.control_changes if cc.number == 1],
                          key=lambda x: x.time)
            e_t, e_v = [cc.time for cc in e_ch], [cc.value for cc in e_ch]
            m_t, m_v = [cc.time for cc in m_ch], [cc.value for cc in m_ch]

            notes = sorted(inst.notes, key=lambda x: (x.start, x.pitch))
            last_n_end_tick = -1

            for i, n in enumerate(notes):
                found    = True
                note_dur = n.end - n.start

                # ✅ DUR: 24단계 = 4분음표 기준
                tempo_idx    = max(0, bisect.bisect_right(tempo_change_times, n.start) - 1)
                bpm          = tempos[tempo_idx] if len(tempos) > 0 else 120.0
                sec_per_beat = 60.0 / bpm
                dur_tick     = max(1, min(192, round((note_dur / sec_per_beat) * 24)))

                # ✅ 박자표 — beats_per_bar 추출
                ts_idx = max(0, bisect.bisect_right(ts_times, n.start) - 1)
                if ts_idx < len(ts_changes):
                    ts            = ts_changes[ts_idx]
                    mkey          = f"{ts.numerator}:{ts.denominator}"
                    meter_tok     = vocab_dict.get(f"METER_{mkey}", vocab_dict["METER_OTHER"])
                    beats_per_bar = ts.numerator
                else:
                    meter_tok     = vocab_dict["METER_OTHER"]
                    beats_per_bar = 4

                # 조성
                k_idx = bisect.bisect_right(key_times, n.start) - 1
                if 0 <= k_idx < len(key_changes):
                    ks    = key_changes[k_idx]
                    root  = ks.key_number % 12
                    mode  = "maj" if ks.key_number < 12 else "min"
                    roots = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
                    rname = FLAT_TO_SHARP.get(roots[root], roots[root])
                    key_tok = vocab_dict.get(f"KEY_{rname}:{mode}", vocab_dict["KEY_NONE"])
                else:
                    key_tok = vocab_dict["KEY_NONE"]

                # ✅ 마디 & TIME — beats_per_bar 기반
                bar_idx        = int(pm.time_to_tick(n.start) // (res * beats_per_bar))
                bar_start_tick = bar_idx * res * beats_per_bar
                rel_tick       = pm.time_to_tick(n.start) - bar_start_tick
                time_tok       = vocab_dict[f"TIME={min(95, rel_tick * 96 // (res * beats_per_bar))}"]

                # ✅ PHRASE_END
                n_start_tick  = pm.time_to_tick(n.start)
                is_phrase_end = (last_n_end_tick > 0 and
                                 (n_start_tick - last_n_end_tick) >= res)

                # ✅ 아티큘레이션
                nxt_start = notes[i+1].start if i+1 < len(notes) else n.end + 1

                mod_val = 0
                if m_t:
                    mi = bisect.bisect_right(m_t, n.start) - 1
                    if mi >= 0: mod_val = m_v[mi]

                if dur_tick <= 2:
                    art_tok = vocab_dict["ART_STACCATO"]
                elif nxt_start == n.start:
                    art_tok = vocab_dict["ART_NORMAL"]
                elif mod_val >= 40:
                    art_tok = vocab_dict["ART_VIBRATO"]
                else:
                    legato_ratio = note_dur / max(nxt_start - n.start, 1e-6)
                    art_tok = (vocab_dict["ART_LEGATO"]
                               if legato_ratio > 0.95
                               else vocab_dict["ART_NORMAL"])

                # EXPR
                expr_val = n.velocity
                if e_t:
                    ei = bisect.bisect_right(e_t, n.start) - 1
                    if ei >= 0: expr_val = e_v[ei]
                expr_tok = vocab_dict[f"EXPR_{min(31, expr_val * 32 // 128)}"]

                vel_tok   = vocab_dict[f"VEL={min(31, n.velocity * 32 // 128)}"]
                pitch_tok = vocab_dict[f"PITCH={n.pitch}"]
                inst_tok  = vocab_dict[f"INST={p}"]

                timeline[bar_idx][p].append(
                    (time_tok, inst_tok, art_tok, expr_tok,
                     pitch_tok, dur_tick, vel_tok,
                     meter_tok, key_tok, is_phrase_end)
                )
                last_n_end_tick = pm.time_to_tick(n.end)

        if not found:
            return None

        # 헤더
        header = [vocab_dict["PIECE_START"], genre_token] + text_tokens

        # ✅ max_bar: 곡 전체 틱을 평균 beats_per_bar로 나눔
        # 대부분 4/4이므로 4 기본값, 마지막 박자표 기준으로 추정
        final_beats = 4
        if ts_changes:
            final_beats = ts_changes[-1].numerator
        max_bar = int(pm.time_to_tick(pm.get_end_time()) // (res * final_beats))

        def get_key_tok_at(bar_time):
            k_idx = bisect.bisect_right(key_times, bar_time) - 1
            if 0 <= k_idx < len(key_changes):
                ks    = key_changes[k_idx]
                root  = ks.key_number % 12
                mode  = "maj" if ks.key_number < 12 else "min"
                roots = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
                rname = FLAT_TO_SHARP.get(roots[root], roots[root])
                return vocab_dict.get(f"KEY_{rname}:{mode}", vocab_dict["KEY_NONE"])
            return vocab_dict["KEY_NONE"]

        def get_meter_tok_at(bar_time):
            ts_idx = max(0, bisect.bisect_right(ts_times, bar_time) - 1)
            if ts_idx < len(ts_changes):
                ts   = ts_changes[ts_idx]
                mkey = f"{ts.numerator}:{ts.denominator}"
                return vocab_dict.get(f"METER_{mkey}", vocab_dict["METER_OTHER"])
            return vocab_dict["METER_OTHER"]

        def get_beats_at(bar_time):
            ts_idx = max(0, bisect.bisect_right(ts_times, bar_time) - 1)
            if ts_idx < len(ts_changes):
                return ts_changes[ts_idx].numerator
            return 4

        body = []
        # ✅ 누적 틱으로 bar_time 정확하게 추적
        accumulated_ticks = 0
        for bar_idx in range(max_bar + 1):
            bar_time      = pm.tick_to_time(accumulated_ticks)
            beats         = get_beats_at(bar_time)
            key_tok       = get_key_tok_at(bar_time)
            meter_tok     = get_meter_tok_at(bar_time)

            if bar_idx in timeline:
                bar_data    = timeline[bar_idx]
                total_notes = sum(len(v) for v in bar_data.values())
                density     = min(5, max(1, total_notes // 4))

                body += [vocab_dict["BAR_START"], key_tok,
                        meter_tok, vocab_dict[f"DENSITY_{density}"]]

                # ✅ 인터리빙: TIME= 기준 정렬
                all_notes = []
                for p in bar_data.keys():
                    for (tt, it, at, et, pt, dt, vt, _, _, phrase_end) in bar_data[p]:
                        time_val = tt - vocab_dict["TIME=0"]
                        all_notes.append((time_val, it, phrase_end,
                                          it, at, et, tt, pt, dt, vt))

                # 시간 순 → 같은 시간이면 악기 ID 순
                all_notes.sort(key=lambda x: (x[0], x[1]))

                for (_, _, phrase_end, it, at, et, tt, pt, dt, vt) in all_notes:
                    if phrase_end:
                        body.append(vocab_dict["PHRASE_END"])
                    body += [it, at, et, tt, pt,
                            vocab_dict[f"DUR={dt}"], vt]

                body.append(vocab_dict["BAR_END"])
            else:
                body += [vocab_dict["BAR_START"], key_tok,
                        meter_tok, vocab_dict["DENSITY_1"],
                        vocab_dict["BAR_END"]]

            accumulated_ticks += res * beats  # ✅ 실제 박자 수만큼 누적

        body.append(vocab_dict["PIECE_END"])
        return header + body

    except Exception:
        return None


def slice_into_chunks(tokens, vocab_dict, chunk_size=CHUNK_SIZE,
                      step=CHUNK_STEP, min_len=MIN_CHUNK):
    bar_start_id = vocab_dict["BAR_START"]

    try:
        first_bar_pos = tokens.index(bar_start_id)
        header = tokens[:first_bar_pos]
        body   = tokens[first_bar_pos:]
    except ValueError:
        return [tokens[:chunk_size]] if len(tokens) >= min_len else []

    bar_positions = [i for i, t in enumerate(body) if t == bar_start_id]
    if not bar_positions:
        full = header + body
        return [full[:chunk_size]] if len(full) >= min_len else []

    header_len           = len(header)
    effective_chunk_size = chunk_size - header_len

    if effective_chunk_size < MIN_BODY_LEN:
        header               = tokens[:2]  # PIECE_START + GENRE만
        body                 = tokens[tokens.index(bar_start_id):]
        bar_positions        = [i for i, t in enumerate(body) if t == bar_start_id]
        effective_chunk_size = chunk_size - len(header)

    chunks = []
    start_bar_idx = 0

    while start_bar_idx < len(bar_positions):
        b_start     = bar_positions[start_bar_idx]
        end_bar_idx = bisect.bisect_right(
            bar_positions, b_start + effective_chunk_size) - 1
        end_bar_idx = max(start_bar_idx, end_bar_idx)
        b_end       = (bar_positions[end_bar_idx + 1]
                       if end_bar_idx + 1 < len(bar_positions)
                       else len(body))

        full_chunk = header + body[b_start:b_end]
        if len(full_chunk) >= min_len:
            chunks.append(full_chunk[:chunk_size])

        next_bar_idx = bisect.bisect_left(bar_positions, b_start + step)
        if next_bar_idx <= start_bar_idx:
            next_bar_idx = start_bar_idx + 1
        start_bar_idx = next_bar_idx

    return chunks

# Execution

In [ ]:
# ✅ [Cell 9] 병렬 전처리 실행기 — 청크 버전
_global_vocab, _global_cap_map = None, None

def _init_worker(v, c):
    global _global_vocab, _global_cap_map
    _global_vocab, _global_cap_map = v, c

def _worker(path):
    try:
        k    = os.path.splitext(os.path.basename(path))[0].lower().strip()
        toks = midi_to_v5_tokens(path, _global_vocab, _global_cap_map.get(k, {}))
        if not toks:
            return []
        chunks = slice_into_chunks(toks, _global_vocab)
        return [json.dumps({"tokens": c}) for c in chunks]
    except Exception:
        return []

def run_parallel_preprocess(file_list, save_name, vocab_dict, cap_map):
    out_path = f"/content/{save_name}.jsonl"
    total_chunks = 0
    with Pool(processes=os.cpu_count(),
              initializer=_init_worker,
              initargs=(vocab_dict, cap_map)) as pool:
        with open(out_path, 'w', encoding='utf-8') as f:
            for results in tqdm(pool.imap_unordered(_worker, file_list),
                                total=len(file_list), desc=save_name):
                for r in results:
                    f.write(r + '\n')
                    total_chunks += 1
    print(f"✅ 저장 완료: {out_path} ({total_chunks:,} 청크)")

In [ ]:
# ✅ [Cell 10] 전처리 실행 — 변경 없음
vocab          = build_v5_vocab()
global_cap_map = load_midicaps_inventory(
    '/content/drive/MyDrive/midi_gpt2_original/midicaps/train.json')

all_midis = glob.glob(
    '/content/home/jhyim0823/GETMusic/lmd_full/**/*.mid', recursive=True)
print(f"✅ 전체 MIDI 파일: {len(all_midis):,}개")

train_files, val_files = train_test_split(
    all_midis, test_size=0.05, random_state=42)
print(f"  Train: {len(train_files):,} / Val: {len(val_files):,}")

run_parallel_preprocess(train_files, "train_pretrain", vocab, global_cap_map)
run_parallel_preprocess(val_files,   "val_pretrain",   vocab, global_cap_map)

✅ MidiCaps 로드 완료: 168,385개
✅ 전체 MIDI 파일: 168,385개
  Train: 159,965 / Val: 8,420


train_pretrain:   0%|          | 0/159965 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI

✅ 저장 완료: /content/train_pretrain.jsonl (3,626,274 청크)


val_pretrain:   0%|          | 0/8420 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI

✅ 저장 완료: /content/val_pretrain.jsonl (191,838 청크)


# Result Validation

In [ ]:
# ✅ [전처리 결과 검증]
# 빈 곡이 없는지, 토큰 분포가 정상인지 확인
# 100토큰 미만 곡은 필터링 (헤더만 있는 빈 껍데기)

import json
from tqdm.auto import tqdm

def validate_dataset(file_path, min_tokens=100):
    total, kept, removed = 0, 0, 0
    lengths = []

    with open(file_path, 'r') as f:
        for line in tqdm(f, desc=f"검증 중: {file_path}"):
            if not line.strip(): continue
            toks = json.loads(line)['tokens']
            total += 1
            if len(toks) >= min_tokens:
                kept += 1
                lengths.append(len(toks))
            else:
                removed += 1

    import numpy as np
    lengths = np.array(lengths)
    print(f"\n{'='*50}")
    print(f"전체: {total:,}곡 | 유효: {kept:,}곡 | 제거: {removed:,}곡")
    print(f"토큰 길이 — 평균: {lengths.mean():.0f} / 중간값: {np.median(lengths):.0f} / 90%: {np.percentile(lengths,90):.0f} / 최대: {lengths.max()}")
    print(f"=> seq_len 권장값: {int(np.percentile(lengths, 75))}")
    print(f"{'='*50}")

validate_dataset("/content/train_pretrain.jsonl")
validate_dataset("/content/val_pretrain.jsonl")

검증 중: /content/train_pretrain.jsonl: 0it [00:00, ?it/s]


전체: 3,626,274곡 | 유효: 3,626,274곡 | 제거: 0곡
토큰 길이 — 평균: 1977 / 중간값: 2048 / 90%: 2048 / 최대: 2048
=> seq_len 권장값: 2048


검증 중: /content/val_pretrain.jsonl: 0it [00:00, ?it/s]


전체: 191,838곡 | 유효: 191,838곡 | 제거: 0곡
토큰 길이 — 평균: 1977 / 중간값: 2048 / 90%: 2048 / 최대: 2048
=> seq_len 권장값: 2048


In [ ]:
import os
path = '/content/train_pretrain.jsonl'
size = os.path.getsize(path) / 1e9
print(f"로컬 파일 크기: {size:.2f} GB")

# Drive 원본도 확인
path2 = '/content/drive/MyDrive/midi_gpt2/processed_data_final_pretrain_inter/train_pretrain.jsonl'
size2 = os.path.getsize(path2) / 1e9
print(f"Drive _inter_ 파일 크기: {size2:.2f} GB")

path3 = '/content/drive/MyDrive/midi_gpt2/processed_data_final_pretrain/train_pretrain.jsonl'
if os.path.exists(path3):
    size3 = os.path.getsize(path3) / 1e9
    print(f"Drive 이전 버전 파일 크기: {size3:.2f} GB")

로컬 파일 크기: 35.12 GB
Drive _inter_ 파일 크기: 35.12 GB
Drive 이전 버전 파일 크기: 35.10 GB


In [ ]:
import json, random

vocab_rev = {v: k for k, v in vocab.items()}
bar_start_id = vocab["BAR_START"]
bar_end_id   = vocab["BAR_END"]

# 랜덤 10개 청크에서 악기 종류 확인
with open('/content/train_pretrain.jsonl') as f:
    lines = f.readlines()

samples = random.sample(lines, 10)

for idx, line in enumerate(samples):
    toks = json.loads(line)['tokens']
    try:
        b_start = toks.index(bar_start_id)
        b_end   = toks.index(bar_end_id, b_start) + 1
        bar     = toks[b_start:b_end]
    except ValueError:
        continue

    inst_toks = [vocab_rev[t] for t in bar if vocab_rev.get(t,'').startswith('INST=')]
    unique_inst = list(dict.fromkeys(inst_toks))  # 순서 유지 중복 제거
    print(f"청크 {idx+1}: {unique_inst}")

청크 1: ['INST=58', 'INST=128', 'INST=33', 'INST=1', 'INST=2', 'INST=89', 'INST=62', 'INST=92']
청크 2: ['INST=128', 'INST=11', 'INST=35', 'INST=40', 'INST=49', 'INST=52', 'INST=25']
청크 3: []
청크 4: ['INST=0', 'INST=11', 'INST=29', 'INST=30', 'INST=37', 'INST=53', 'INST=128']
청크 5: ['INST=0', 'INST=128', 'INST=32', 'INST=49']
청크 6: ['INST=32', 'INST=128', 'INST=52', 'INST=8', 'INST=73', 'INST=56']
청크 7: ['INST=25', 'INST=35', 'INST=38', 'INST=50', 'INST=128', 'INST=75']
청크 8: ['INST=0', 'INST=40', 'INST=41', 'INST=42']
청크 9: ['INST=58', 'INST=60', 'INST=61', 'INST=70', 'INST=128', 'INST=27']
청크 10: ['INST=4', 'INST=32', 'INST=66', 'INST=128', 'INST=24']


In [ ]:
# 청크 2 첫 번째 BAR 전체 디코딩
toks = json.loads(samples[1])['tokens']
try:
    b_start = toks.index(bar_start_id)
    b_end   = toks.index(bar_end_id, b_start) + 1
    bar     = toks[b_start:b_end]
    print(f"BAR 토큰 수: {len(bar)}")
    print()
    for t in bar:
        name = vocab_rev.get(t, '?')
        # INST, TIME, PITCH만 강조해서 출력
        if any(name.startswith(x) for x in ['INST=','TIME=','PITCH=','BAR','KEY','METER','DENSITY']):
            print(name)
except ValueError:
    print("없음")

BAR 토큰 수: 223

BAR_START
KEY_NONE
METER_OTHER
DENSITY_5
INST=128
TIME=4
PITCH=42
INST=128
TIME=4
PITCH=63
INST=11
TIME=21
PITCH=70
INST=35
TIME=21
PITCH=31
INST=40
TIME=21
PITCH=70
INST=49
TIME=21
PITCH=70
INST=52
TIME=21
PITCH=67
INST=128
TIME=21
PITCH=36
INST=128
TIME=21
PITCH=42
INST=25
TIME=38
PITCH=55
INST=25
TIME=38
PITCH=58
INST=25
TIME=38
PITCH=62
INST=128
TIME=38
PITCH=42
INST=128
TIME=46
PITCH=46
INST=128
TIME=46
PITCH=63
INST=128
TIME=55
PITCH=42
INST=128
TIME=55
PITCH=54
INST=128
TIME=55
PITCH=62
INST=11
TIME=72
PITCH=70
INST=25
TIME=72
PITCH=55
INST=25
TIME=72
PITCH=58
INST=25
TIME=72
PITCH=62
INST=35
TIME=72
PITCH=31
INST=128
TIME=72
PITCH=36
INST=128
TIME=72
PITCH=42
INST=11
TIME=88
PITCH=67
INST=35
TIME=88
PITCH=31
INST=40
TIME=88
PITCH=67
INST=49
TIME=88
PITCH=67
INST=128
TIME=88
PITCH=36
INST=128
TIME=88
PITCH=42
BAR_END


In [ ]:
# ✅ 토큰 분포 검증 (전처리 후 필수 실행)
import json
from collections import Counter
from tqdm.auto import tqdm

vocab_rev = {v: k for k, v in vocab.items()}
token_counter = Counter()

with open('/content/train_pretrain.jsonl') as f:
    for line in tqdm(f, desc="분포 분석"):
        toks = json.loads(line)['tokens']
        token_counter.update(toks)

total = sum(token_counter.values())

# 핵심 지표만 출력
print("=== 경고 지표 ===")
for key in ['KEY_NONE', 'METER_OTHER', 'GENRE_UNKNOWN', 'ART_NORMAL']:
    tid = vocab.get(key)
    if tid:
        ratio = token_counter[tid] / total
        flag = "⚠️" if ratio > 0.7 else "✅"
        print(f"{flag} {key}: {ratio:.1%}")

print("\n=== 샘플 디코딩 (첫 곡 50토큰) ===")
with open('/content/train_pretrain.jsonl') as f:
    sample = json.loads(f.readline())['tokens']
print(' '.join(vocab_rev.get(t, '?') for t in sample[:50]))

print("\n=== Top 20 토큰 ===")
for tid, cnt in token_counter.most_common(20):
    print(f"{vocab_rev.get(tid,'?'):<20} {cnt/total:.2%}")

분포 분석: 0it [00:00, ?it/s]

=== 경고 지표 ===
✅ KEY_NONE: 0.2%
✅ METER_OTHER: 0.0%
✅ GENRE_UNKNOWN: 0.0%
✅ ART_NORMAL: 7.5%

=== 샘플 디코딩 (첫 곡 50토큰) ===
PIECE_START GENRE_POP TEXT_piano BAR_START KEY_C:maj METER_6:8 DENSITY_1 INST=0 ART_LEGATO EXPR_17 TIME=0 PITCH=64 DUR=34 VEL=17 INST=0 ART_LEGATO EXPR_19 TIME=24 PITCH=62 DUR=34 VEL=19 INST=0 ART_LEGATO EXPR_18 TIME=48 PITCH=60 DUR=68 VEL=18 BAR_END BAR_START KEY_C:maj METER_6:8 DENSITY_1 INST=0 ART_LEGATO EXPR_20 TIME=0 PITCH=64 DUR=34 VEL=20 INST=0 ART_LEGATO EXPR_18 TIME=24 PITCH=62 DUR=34 VEL=18 INST=0 ART_LEGATO EXPR_18

=== Top 20 토큰 ===
ART_NORMAL           7.47%
EXPR_31              5.13%
ART_LEGATO           4.09%
INST=128             4.05%
ART_STACCATO         2.35%
TIME=0               1.88%
INST=0               1.66%
DUR=1                1.64%
DUR=6                1.55%
VEL=25               1.48%
DUR=12               1.40%
VEL=31               1.39%
EXPR_25              1.28%
TIME=48              1.25%
TIME=24              1.18%
TIME=72              1.12%


# Save Jsonl Files

In [ ]:
# ✅ [Drive 백업]
# 전처리 완료된 JSONL을 Drive에 저장
# Colab 런타임이 끊겨도 데이터 보존됨

import shutil, os

SAVE_DIR = '/content/drive/MyDrive/midi_gpt2/processed_data_final_pretrain_inter/'
os.makedirs(SAVE_DIR, exist_ok=True)

for fname in ["train_pretrain.jsonl", "val_pretrain.jsonl"]:
    src = f"/content/{fname}"
    dst = os.path.join(SAVE_DIR, fname)
    shutil.copy2(src, dst)
    print(f"✅ 저장: {dst}")

✅ 저장: /content/drive/MyDrive/midi_gpt2/processed_data_final_pretrain_inter/train_pretrain.jsonl
✅ 저장: /content/drive/MyDrive/midi_gpt2/processed_data_final_pretrain_inter/val_pretrain.jsonl


In [ ]:
import os

path = '/content/drive/MyDrive/midi_gpt2/processed_data_final_pretrain_inter/train_pretrain.jsonl'
if os.path.exists(path):
    size = os.path.getsize(path)
    print(f"✅ 파일 존재: {size/1e9:.2f} GB")
else:
    print("❌ 파일 없음")

✅ 파일 존재: 35.12 GB


# Pre-Train
-------

# Environments & Imports

In [ ]:
# 1. 모든 충돌 패키지 완전 제거
!pip uninstall -y torch torchvision torchaudio transformers accelerate datasets peft sentence-transformers torchtune -q

# 2. 캐시 완전 삭제
!pip cache purge

# 3. PyTorch 생태계 재설치 (torch 2.5.1 + cu121)
!pip install "torch==2.5.1+cu121" "torchvision==0.20.1+cu121" "torchaudio==2.5.1+cu121" \
  --index-url https://download.pytorch.org/whl/cu121 --no-cache-dir

# 4. flash-attn wheel (빌드 없이)
!pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl" --no-cache-dir

# 5. 최신 호환 버전들 (yanked 피함)
!pip install transformers==4.51.3 accelerate==1.2.1 datasets==3.0.1 --no-cache-dir

Files removed: 11
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 366.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 148.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 332.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 413.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 406.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 373.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 394.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 313.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 403.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 226.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 302.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
print(f"✅ PyTorch: {torch.__version__}")

import flash_attn
print("✅ FlashAttention-2")

from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
print("✅ Transformers + Trainer")

print("🎉 모든 패키지 정상!")


✅ PyTorch: 2.5.1+cu121
✅ FlashAttention-2
✅ Transformers + Trainer
🎉 모든 패키지 정상!


In [ ]:
# ===== Cell 18: 학습 임포트 및 공통 설정 =====
import random, torch, time, subprocess, shutil
import torch.nn as nn
from torch.utils.data import IterableDataset
from torch.nn import CrossEntropyLoss
from transformers import AutoConfig, AutoModelForCausalLM, TrainingArguments, Trainer
import subprocess

SEQ_LEN    = 2048
VOCAB_SIZE = 682
PAD_ID     = vocab["PAD"]
KEY_IDS    = {v for k,v in vocab.items()
              if k.startswith("KEY_") and k != "KEY_NONE"}
GENRE_IDS  = {v for k,v in vocab.items()
              if k.startswith("GENRE_")}

PRETRAIN_CKPT = '/content/drive/MyDrive/midi_gpt2/checkpoints/pretrain_0.5b'
os.makedirs(PRETRAIN_CKPT, exist_ok=True)

print(f"✅ PAD_ID: {PAD_ID}")
print(f"   KEY 토큰: {len(KEY_IDS)}개")
print(f"   GENRE 토큰: {len(GENRE_IDS)}개")

✅ PAD_ID: 0
   KEY 토큰: 24개
   GENRE 토큰: 7개


# DriveToLocal

In [ ]:
# ===== Cell 19: Drive → 로컬 복사 (SSD 업로드) =====
import time, shutil, os

def copy_if_needed(src, dst):
    if not os.path.exists(src):
        print(f"❌ 원본 없음: {src}")
        return
    if os.path.exists(dst):
        print(f"✅ 이미 존재: {dst} ({os.path.getsize(dst)/1e9:.1f}GB)")
        return
    print(f"📦 복사 중: {os.path.basename(src)} ...")
    start = time.time()
    shutil.copy2(src, dst)
    print(f"✅ 완료: {os.path.getsize(dst)/1e9:.1f}GB / {time.time()-start:.0f}초")

# # 1. 기존 JSONL 파일 복사
# copy_if_needed(
#     '/content/drive/MyDrive/midi_gpt2/processed_data_final_pretrain_inter/train_pretrain.jsonl',
#     '/content/train_pretrain.jsonl')
# copy_if_needed(
#     '/content/drive/MyDrive/midi_gpt2/processed_data_final_pretrain_inter/val_pretrain.jsonl',
#     '/content/val_pretrain.jsonl')

# 2. ✅ 추가: BIN 파일 복사 (로컬 SSD로 업로드)
copy_if_needed(
    '/content/drive/MyDrive/midi_gpt2/train_pretrain.bin',
    '/content/train_pretrain.bin')
copy_if_needed(
    '/content/drive/MyDrive/midi_gpt2/val_pretrain.bin',
    '/content/val_pretrain.bin')

📦 복사 중: train_pretrain.bin ...
✅ 완료: 14.9GB / 189초
📦 복사 중: val_pretrain.bin ...
✅ 완료: 0.8GB / 14초


# Load Model

In [ ]:
# ===== Cell 20: 모델 로드 (Qwen2.5-0.5B + Embedding 교체) =====
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'  # 1.5B → 0.5B

config = AutoConfig.from_pretrained(MODEL_NAME)
config.vocab_size              = VOCAB_SIZE
config.pad_token_id            = PAD_ID
config.max_position_embeddings = SEQ_LEN
config.sliding_window          = None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    config                  = config,
    torch_dtype             = torch.bfloat16,
    ignore_mismatched_sizes = True,
    attn_implementation     = "flash_attention_2",  # ✅ 추가
)

model.config.use_cache = False
model.model.embed_tokens = nn.Embedding(
    VOCAB_SIZE, config.hidden_size).to(torch.bfloat16)
model.lm_head = nn.Linear(
    config.hidden_size, VOCAB_SIZE, bias=False).to(torch.bfloat16)
# nn.init.normal_(model.model.embed_tokens.weight, mean=0.0, std=0.02)
# nn.init.normal_(model.lm_head.weight,            mean=0.0, std=0.02)

print(f"✅ 모델 로드 완료")
print(f"   파라미터: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B and are newly initialized because the shapes did not match:
- model.embed_tokens.weight: found shape torch.Size([151936, 896]) in the checkpoint and torch.Size([682, 896]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

✅ 모델 로드 완료
   파라미터: 359.1M
   GPU: NVIDIA A100-SXM4-80GB
   GPU RAM: 85.1GB


# Load Dataset

In [ ]:
# ===== Cell 21: Binary 로드 + Dataset (최종 최적화 버전) =====
import numpy as np
import torch
import os, json, random, time, shutil, subprocess
from torch.utils.data import Dataset
from tqdm.auto import tqdm

# 상수 설정
PRE_ID       = vocab["<PRE>"]
SUF_ID       = vocab["<SUF>"]
MID_ID       = vocab["<MID>"]
BAR_START_ID = vocab["BAR_START"]
PAD_ID       = vocab.get("<PAD>", 0) # PAD_ID 확인 필요
FIM_RATE     = 0.5

def apply_fim_online(tokens, seq_len):
    if random.random() > FIM_RATE:
        return tokens
    try:
        first_bar = tokens.index(BAR_START_ID)
    except ValueError:
        return tokens

    header        = tokens[:first_bar]
    body_tokens   = tokens[first_bar:]
    bar_positions = [i for i, t in enumerate(body_tokens) if t == BAR_START_ID]

    if len(bar_positions) < 3:
        return tokens

    mid_idx   = random.randint(1, len(bar_positions) - 2)
    mid_start = bar_positions[mid_idx]
    mid_end   = (bar_positions[mid_idx + 1]
                 if mid_idx + 1 < len(bar_positions)
                 else len(body_tokens))

    prefix = body_tokens[:mid_start]
    middle = body_tokens[mid_start:mid_end]
    suffix = body_tokens[mid_end:]

    fim = [PRE_ID] + header + prefix + [SUF_ID] + suffix + [MID_ID] + middle
    return fim[:seq_len]

def prepare_binary_with_fallback(local_path, drive_path, jsonl_path, seq_len=2048):
    """
    1. 로컬 SSD 확인
    2. 드라이브 복사 확인
    3. 없으면 JSONL에서 신규 변환
    """
    # 1. 로컬 SSD에 이미 있는 경우
    if os.path.exists(local_path):
        n = os.path.getsize(local_path) // (seq_len * 2) # uint16 = 2bytes
        print(f"✅ 로컬 SSD 활용: {os.path.basename(local_path)} ({n:,} samples)")
        return n

    # 2. 로컬에는 없지만 드라이브에 있는 경우
    if os.path.exists(drive_path):
        print(f"📦 드라이브에서 로컬 SSD로 복사 중... ({os.path.basename(drive_path)})")
        start = time.time()
        shutil.copy2(drive_path, local_path)
        n = os.path.getsize(local_path) // (seq_len * 2)
        print(f"✅ 복사 완료: {time.time()-start:.0f}초 ({n:,} samples)")
        return n

    # 3. 둘 다 없으면 신규 변환 (기존 jsonl_to_binary 로직)
    print(f"⚠️ 바이너리 없음. {os.path.basename(jsonl_path)}에서 변환 시작...")
    return jsonl_to_binary(jsonl_path, local_path, seq_len)

# 기존 jsonl_to_binary는 fallback용으로 유지
def jsonl_to_binary(jsonl_path, bin_path, seq_len=2048):
    if not os.path.exists(jsonl_path):
        raise FileNotFoundError(f"❌ 원본 JSONL 없음: {jsonl_path}")

    with open(jsonl_path) as f:
        n = sum(1 for line in f if line.strip())

    arr = np.memmap(bin_path, dtype='uint16', mode='w+', shape=(n, seq_len))
    with open(jsonl_path) as f:
        for idx, line in enumerate(tqdm(f, total=n, desc="변환")):
            toks = json.loads(line)['tokens']
            toks = (toks + [PAD_ID] * seq_len)[:seq_len]
            arr[idx] = np.array(toks, dtype='uint16')
    arr.flush()
    return n

class MidiBinaryDataset(Dataset):
    def __init__(self, bin_path, n_samples, seq_len):
        self.seq_len = seq_len
        self.n       = n_samples
        self.arr     = np.memmap(bin_path, dtype='uint16', mode='r', shape=(n_samples, seq_len))

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        toks = self.arr[idx].tolist()
        toks = apply_fim_online(toks, self.seq_len)
        toks = (toks + [PAD_ID] * self.seq_len)[:self.seq_len]
        return torch.tensor(toks, dtype=torch.long)

def midi_collate_fn(batch):
    ids    = torch.stack(batch)
    labels = ids.clone()
    labels[labels == PAD_ID] = -100
    mask   = (ids != PAD_ID).long()
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}

# --- 실행부 ---
# 경로 정의 (연구원님 환경에 맞게 수정)
DRIVE_DIR = '/content/drive/MyDrive/midi_gpt2'
TRAIN_BIN_DRIVE = f'{DRIVE_DIR}/train_pretrain.bin'
VAL_BIN_DRIVE   = f'{DRIVE_DIR}/val_pretrain.bin'

train_n = prepare_binary_with_fallback('/content/train_pretrain.bin', TRAIN_BIN_DRIVE, '/content/train_pretrain.jsonl', SEQ_LEN)
val_n   = prepare_binary_with_fallback('/content/val_pretrain.bin',   VAL_BIN_DRIVE,   '/content/val_pretrain.jsonl',   SEQ_LEN)

train_ds = MidiBinaryDataset('/content/train_pretrain.bin', train_n, SEQ_LEN)
val_ds   = MidiBinaryDataset('/content/val_pretrain.bin',   val_n,   SEQ_LEN)

print(f"\n🚀 준비 완료! Train: {len(train_ds):,}개 / Val: {len(val_ds):,}개")

✅ 로컬 SSD 활용: train_pretrain.bin (3,626,274 samples)
✅ 로컬 SSD 활용: val_pretrain.bin (191,838 samples)

🚀 준비 완료! Train: 3,626,274개 / Val: 191,838개


# WeightedTrainer

In [ ]:
# ===== Cell 22: WeightedTrainer + tqdm 진행바 =====
from transformers import TrainerCallback, TrainerState, TrainerControl
from tqdm.auto import tqdm as tqdm_auto
import shutil, os

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fct     = CrossEntropyLoss(reduction='none', ignore_index=-100)
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = loss_fct(
            shift_logits.view(-1, VOCAB_SIZE),
            shift_labels.view(-1))

        weights = torch.ones_like(shift_labels, dtype=torch.float)
        for kid in KEY_IDS:   weights[shift_labels == kid] = 3.0
        for gid in GENRE_IDS: weights[shift_labels == gid] = 2.0

        w = weights.view(-1)
        w[shift_labels.view(-1) == -100] = 0.0
        weighted_loss = (loss * w).sum() / max(w.nonzero().numel(), 1)
        return (weighted_loss, outputs) if return_outputs else weighted_loss

# **Pretrain Execution**

In [ ]:
# ===== Cell 22.5: 콜백 정의 =====
from transformers import TrainerCallback
from tqdm.auto import tqdm as tqdm_auto
import shutil, os

class TqdmProgressCallback(TrainerCallback):
    def __init__(self, total_steps):
        self.total_steps = total_steps
        self.pbar        = None
        self.cur_loss    = 0.0
        self.cur_lr      = 0.0
        self.patience    = 0
        self.max_patience = 0

    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm_auto(
            total       = self.total_steps,
            desc        = "Pretrain",
            dynamic_ncols= True,
        )
        # EarlyStoppingCallback의 patience 찾기
        for cb in kwargs.get('callbacks', []):
            if hasattr(cb, 'early_stopping_patience'):
                self.max_patience = cb.early_stopping_patience
                break

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or self.pbar is None:
            return
        if "loss" in logs:
            self.cur_loss = logs["loss"]
        if "learning_rate" in logs:
            self.cur_lr = logs["learning_rate"]
        # tqdm postfix 업데이트
        self.pbar.set_postfix_str(
            f"loss={self.cur_loss:.4f}, "
            f"lr={self.cur_lr:.2e}, "
            f"patience={self.patience}/{self.max_patience}"
        )
        self.pbar.n = state.global_step
        self.pbar.refresh()

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()


class SaveBestCallback(TrainerCallback):
    def __init__(self, best_dir, output_dir, max_patience=10):
        self.best_dir      = best_dir
        self.output_dir    = output_dir
        self.best_loss     = float('inf')
        self.best_step     = 0
        self.patience_cnt  = 0
        self.max_patience  = max_patience
        self._tqdm_cb      = None
        self._pending_save = False

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        eval_loss = metrics.get("eval_loss", float('inf'))
        step      = state.global_step

        if eval_loss < self.best_loss:
            self.best_loss     = eval_loss
            self.best_step     = step
            self.patience_cnt  = 0
            self._pending_save = True
            tqdm_auto.write(f"\n  Step {step}: val_loss={eval_loss:.4f} (best 갱신 — 다음 save에서 복사)")
        else:
            self.patience_cnt += 1
            tqdm_auto.write(
                f"\n  Step {step}: val_loss={eval_loss:.4f} "
                f"(best={self.best_loss:.4f}, patience={self.patience_cnt}/{self.max_patience})"
            )

        if self._tqdm_cb:
            self._tqdm_cb.patience = self.patience_cnt

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        tqdm_auto.write(f"  ✓ Checkpoint step {step}")

        if self._pending_save:
            # eval과 save가 같은 step이거나 save가 1 뒤에 오는 경우 모두 처리
            ckpt_path = os.path.join(self.output_dir, f"checkpoint-{step}")
            if os.path.exists(ckpt_path):
                os.makedirs(os.path.dirname(self.best_dir), exist_ok=True)
                if os.path.exists(self.best_dir):
                    shutil.rmtree(self.best_dir)
                shutil.copytree(ckpt_path, self.best_dir)
                tqdm_auto.write(f"  ✓ Best model saved → {self.best_dir}  (val_loss={self.best_loss:.4f})")
                self._pending_save = False
            else:
                tqdm_auto.write(f"  ⚠ checkpoint-{step} 없음")

    def on_train_end(self, args, state, control, **kwargs):
        tqdm_auto.write(
            f"\nPretrain 완료! Best val_loss: {self.best_loss:.4f} "
            f"(step {self.best_step})"
        )

In [ ]:
import torch
import numpy as np

# PyTorch 보안 검문소에 안전한 객체들 등록
torch.serialization.add_safe_globals([
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtype,
    np.dtypes.UInt32DType,  # <--- 에러의 주범을 명단에 추가
    # 만약 위 코드가 안 되면 아래 구형 경로도 추가하십시오
    # np._core.multiarray._reconstruct
])

print("✅ 보안 패치 완료: 이제 RNG 상태를 안전하게 로드합니다.")

✅ 보안 패치 완료: 이제 RNG 상태를 안전하게 로드합니다.


In [ ]:
# ===== Cell 23: Pretrain 실행 =====
from transformers import EarlyStoppingCallback
import os
import torch.serialization

import numpy as np
torch.serialization.add_safe_globals([np.core.multiarray._reconstruct, np.ndarray, np.dtype])

BEST_CKPT = PRETRAIN_CKPT + "/best"

total_steps = (len(train_ds) // (16 * 2)) * 3  # steps/epoch × epochs

args = TrainingArguments(
    output_dir                  = PRETRAIN_CKPT,
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = 2,
    learning_rate               = 5e-5,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    bf16                        = True,
    eval_strategy               = "steps",
    eval_steps                  = 2000,
    save_strategy               = "steps",
    save_steps                  = 2000,
    save_total_limit = 2,                    # ✅ latest 1개만 유지
    logging_steps               = 200,
    dataloader_num_workers      = 4,
    dataloader_drop_last        = True,
    report_to                   = "none",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    disable_tqdm                = True,      # ✅ 기본 tqdm 끄고 커스텀 사용
    ignore_data_skip            = True,
    remove_unused_columns       = False,
    resume_from_checkpoint      = "/content/drive/MyDrive/midi_gpt2/checkpoints/pretrain_0.5b/checkpoint-194000",
  )

def get_last_checkpoint(ckpt_dir):
    if not os.path.exists(ckpt_dir):
        return None
    ckpts = [d for d in os.listdir(ckpt_dir)
             if d.startswith("checkpoint-")]
    if not ckpts:
        return None
    last = sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1]
    path = os.path.join(ckpt_dir, last)
    print(f"🔄 Resume from: {path}")
    return path

resume_from = "/content/drive/MyDrive/midi_gpt2/checkpoints/pretrain_0.5b/checkpoint-194000"

# patience 공유를 위해 콜백 연결
PATIENCE = 10
save_best_cb = SaveBestCallback(
    best_dir=BEST_CKPT,
    output_dir=PRETRAIN_CKPT,
    max_patience=PATIENCE,
)
tqdm_cb      = TqdmProgressCallback(total_steps=total_steps)
save_best_cb._tqdm_cb = tqdm_cb  # patience 실시간 동기화
tqdm_cb.max_patience  = PATIENCE

trainer = WeightedTrainer(
    model         = model,
    args          = args,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    data_collator = midi_collate_fn,
    callbacks     = [
        EarlyStoppingCallback(early_stopping_patience=PATIENCE),
        save_best_cb,
        tqdm_cb,
    ],
)

print("🚀 Pretrain 시작")
print(f"   모델: Qwen2.5-0.5B")
print(f"   effective batch: {16 * 2}")
print(f"   steps/epoch: {len(train_ds) // (16 * 2):,}")
print(f"   총 steps: {total_steps:,}")
print(f"   eval/save: 매 2000 steps")
print(f"   early stopping patience: 10")
print(f"   best 저장: {BEST_CKPT}")
print(f"   resume: {resume_from if resume_from else '없음 (처음부터)'}")

trainer.train(resume_from_checkpoint=resume_from)

# 학습 완료 후 best → final 복사
if os.path.exists(BEST_CKPT):
    final_path = PRETRAIN_CKPT + "/final"
    if os.path.exists(final_path):
        shutil.rmtree(final_path)
    shutil.copytree(BEST_CKPT, final_path)
    print(f"✅ Final 저장: {final_path}")
    # Early stopping 메시지
    if trainer.state.global_step < total_steps:
        tqdm_auto.write(
            f"\n🛑 Early Stopping! {PATIENCE}번 연속 개선 없음"
        )

print("✅ Pretrain 완료")

🚀 Pretrain 시작
   모델: Qwen2.5-0.5B
   effective batch: 32
   steps/epoch: 113,321
   총 steps: 339,963
   eval/save: 매 2000 steps
   early stopping patience: 10
   best 저장: /content/drive/MyDrive/midi_gpt2/checkpoints/pretrain_0.5b/best
   resume: /content/drive/MyDrive/midi_gpt2/checkpoints/pretrain_0.5b/checkpoint-194000


Pretrain:   0%|          | 0/339963 [00:00<?, ?it/s]

{'loss': 0.6452, 'grad_norm': 0.890625, 'learning_rate': 0.0001271550134221943, 'epoch': 1.0017648979447764}
{'loss': 0.6312, 'grad_norm': 0.765625, 'learning_rate': 0.0001268666398132323, 'epoch': 1.0035297958895526}
{'loss': 0.6359, 'grad_norm': 0.78125, 'learning_rate': 0.00012657835376111222, 'epoch': 1.005294693834329}
{'loss': 0.6278, 'grad_norm': 0.79296875, 'learning_rate': 0.0001262901563569604, 'epoch': 1.0070595917791054}
{'loss': 0.6314, 'grad_norm': 0.84375, 'learning_rate': 0.0001260020486915675, 'epoch': 1.0088244897238816}
{'loss': 0.6421, 'grad_norm': 0.7734375, 'learning_rate': 0.0001257140318553847, 'epoch': 1.010589387668658}
{'loss': 0.6384, 'grad_norm': 0.89453125, 'learning_rate': 0.00012542610693851917, 'epoch': 1.0123542856134344}
{'loss': 0.6225, 'grad_norm': 0.91015625, 'learning_rate': 0.0001251382750307303, 'epoch': 1.0141191835582108}
{'loss': 0.6183, 'grad_norm': 0.7578125, 'learning_rate': 0.00012485053722142555, 'epoch': 1.015884081502987}
{'loss': 0.62

# **Fine-Tune**
------

# Preprocessing For Fine-Tune